# T45E / E11 — Chi phí phát hiện khi mô hình sinh cũng là mô hình đọc

**Câu hỏi.** Khi một mô hình vừa sinh câu trả lời vừa cung cấp trọng số chú ý (kịch bản tự kiểm tra),
việc phát hiện làm tốn thêm bao nhiêu thời gian? Trước đây mục này chỉ là lập luận, vì Qwen2.5-7B tràn số
ở `float16` nên không sinh được trên T4. Qwen2.5-3B không có lớp nào tràn số nên đảm nhận được cả hai vai.

**Cách đo.** 100 ngữ cảnh của tập test ViHallu (seed 42). Câu trả lời được sinh tự do một lần, rồi **phát
lại đúng chuỗi token đó** theo bốn cách, để mọi cách làm cùng một khối lượng tính toán:

| Cách | Cài đặt chú ý | Đọc chú ý | Ý nghĩa |
|---|---|---|---|
| A | SDPA | không | chi phí sinh hiện nay |
| B | eager | không | B − A: chi phí tắt cài đặt tối ưu |
| C | eager | hook, ngay khi sinh | C − B: chi phí của việc phát hiện; C − A: toàn bộ phần thêm |
| R | eager | hook, đọc lại một lượt | chi phí nếu phải đọc lại sau khi sinh |

Mỗi cách chạy hai lượt mỗi mẫu, thứ tự xoay vòng và đảo ngược, để nếu card chậm dần thì chậm đều cả bốn.

**Dự đoán ghi trước khi chạy** nằm trong `PREDICTIONS` của `scripts/measure_self_check.py`.

**Chạy trong một phiên T4 riêng.** Ước tính 1,5–2 giờ. Bị ngắt giữa chừng thì chạy lại ô 6: các mẫu đã đo
không đo lại. Dùng *Save Version* (Save & Run All) là đủ.

In [ ]:
# Ô 1 — lấy mã nguồn. Chạy lại được nhiều lần.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

In [ ]:
# Ô 2 — cài đặt, ghim phiên bản theo uv.lock.
!pip install -q --no-deps -e .
!pip install -q -r requirements-kaggle.txt

In [ ]:
# Ô 3 — TIỀN KIỂM. Vài giây, chạy trước mọi thứ.
import importlib.util
import sys
from pathlib import Path

import torch

sys.path.insert(0, "src")
from vihallulens.data.paths import find_raw_dir

problems = []
absent = [n for n in ("torch", "transformers", "bitsandbytes", "accelerate", "pandas")
          if importlib.util.find_spec(n) is None]
if absent:
    problems.append(f"thiếu gói {absent}")
if not torch.cuda.is_available():
    problems.append("chưa bật GPU T4")
else:
    print("GPU:", torch.cuda.get_device_name(0))
try:
    raw = Path(find_raw_dir())
    missing = [n for n in ['vihallu_train.csv'] if not (raw / n).is_file()]
    print("dữ liệu thô:", raw, "— thiếu", missing or "không")
    if missing:
        problems.append(f"thiếu tệp {missing}")
except Exception as error:
    problems.append(f"chưa gắn dataset dữ liệu thô ({error})")
if problems:
    raise SystemExit("TIỀN KIỂM HỎNG: " + "; ".join(problems))
print("Tiền kiểm đạt.")

In [ ]:
# Ô 4 — chuẩn bị dữ liệu và kiểm môi trường. Khoảng 2 phút, CPU.
!python scripts/probe_env.py
!python scripts/normalize_data.py --dataset vihallu
!python scripts/split_data.py --only vihallu
!python -m pytest tests/test_incremental.py tests/test_self_check.py tests/test_revisions.py -q

In [ ]:
# Ô 5 — chạy thử 2 mẫu. Khoảng 5 phút gồm tải mô hình. Đọc dòng "thử float16": logit phải hữu hạn
# và câu trả lời phải là tiếng Việt bình thường. Nếu float16 hỏng, script tự chuyển sang bfloat16.
!python scripts/measure_self_check.py run --limit 2 --out /kaggle/working/e11_self_check.jsonl
!python scripts/measure_self_check.py report --path /kaggle/working/e11_self_check.jsonl

In [ ]:
# Ô 6 — đủ 100 mẫu. In tiến độ mỗi 5 mẫu.
!python scripts/measure_self_check.py run --out /kaggle/working/e11_self_check.jsonl

In [ ]:
# Ô 7 — tổng hợp. Tải về hai tệp trong /kaggle/working: e11_self_check.jsonl và e11_self_check.json.
!python scripts/measure_self_check.py report --path /kaggle/working/e11_self_check.jsonl --json /kaggle/working/e11_self_check.json